# 02 - Connect VPN

Generate VPN client profile and verify private DNS resolution over the active VPN connection.


## Overview

End state: laptop connected with Azure VPN Client and service hostnames resolving to private endpoint IPs.


In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv("../env/.env")
gateway = os.getenv("VNET_GATEWAY_NAME")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
if not gateway or not resource_group:
    raise RuntimeError("Missing VNET_GATEWAY_NAME or AZURE_RESOURCE_GROUP in env/.env")

out_dir = Path("../outputs/vpn-client")
out_dir.mkdir(parents=True, exist_ok=True)
zip_path = out_dir / "vpnclient.zip"
profile_url = subprocess.run(["az", "network", "vnet-gateway", "vpn-client", "generate", "--resource-group", resource_group, "--name", gateway, "--authentication-method", "AADAuthentication", "--processor-architecture", "Amd64", "--query", "profileUrl", "-o", "tsv"], capture_output=True, text=True, check=True).stdout.strip()
if not profile_url:
    raise RuntimeError("No VPN profile URL returned.")
response = requests.get(profile_url, timeout=120)
response.raise_for_status()
zip_path.write_bytes(response.content)
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(out_dir)
xml_path = out_dir / "AzureVPN" / "azurevpnconfig.xml"
if not xml_path.exists():
    raise FileNotFoundError(xml_path)
print(f"VPN profile XML: {xml_path.resolve()}")


## Install Azure VPN Client

- Windows: install from Microsoft Store (Windows 10 build 17763+).
- macOS: install Azure VPN Client from App Store.


## Import the profile

1. Open Azure VPN Client.
2. Select **+** then **Import**.
3. Choose `azurevpnconfig.xml` generated above.
4. Save the profile.


## Edit custom DNS

Set profile DNS server to `DNS_FORWARDER_PRIVATE_IP` from `env/.env`.

Without this, `*.privatelink.*` names resolve publicly and later notebooks fail.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../env/.env")
dns_ip = os.getenv("DNS_FORWARDER_PRIVATE_IP", "")
if not dns_ip:
    raise RuntimeError("DNS_FORWARDER_PRIVATE_IP missing in env/.env")
print(f"Custom DNS to configure in Azure VPN Client profile: {dns_ip}")


## Connect

Click **Connect**, sign in with AAD, and wait for profile state **Connected**.


In [ ]:
import ipaddress
import os
import socket
from dotenv import load_dotenv

load_dotenv("../env/.env")
storage_name = os.getenv("STORAGE_ACCOUNT_NAME")
search_name = os.getenv("SEARCH_SERVICE_NAME")
pe_prefix = os.getenv("SNET_PE_PREFIX")
if not storage_name or not search_name or not pe_prefix:
    raise RuntimeError("Missing STORAGE_ACCOUNT_NAME, SEARCH_SERVICE_NAME, or SNET_PE_PREFIX in env/.env")
network = ipaddress.ip_network(pe_prefix)
targets = [f"{storage_name}.blob.core.windows.net", f"{search_name}.search.windows.net"]
for host in targets:
    ip = socket.gethostbyname(host)
    print(f"{host} -> {ip}")
    if ipaddress.ip_address(ip) not in network:
        raise RuntimeError(f"{host} resolved to {ip}, not in {pe_prefix}. Revisit Edit custom DNS step and ensure VPN is connected.")
print("✓ DNS verification passed")


## Troubleshooting

- AAD sign-in popup blocked: allow popups and retry.
- Public IP resolution: custom DNS not set or VPN disconnected.
- Consent error: run consent check from `00_setup.ipynb`.
- Drops under concurrency: gateway SKU may need scaling.


---

Continue with `03_configure.ipynb`.
